In [1]:
from data_preparation import OzonDataFormer, DatasetType

In [2]:
dataformer = OzonDataFormer()

In [3]:
train_data = dataformer.construct_dataset(DatasetType.TRAIN)

In [4]:
train_data = train_data.to_pandas()
X = train_data.drop(columns=['user_id', 'target'])
y = train_data['target']

In [5]:
mask = X.index % 10 <= 6

In [6]:
X_train = X.loc[mask, :]
X_val = X.loc[~mask, :]
y_train = y.loc[mask]
y_val = y.loc[~mask]

In [8]:
from catboost import CatBoostClassifier, Pool
from catboost.utils import get_gpu_device_count

In [9]:
cat_features = list(X.select_dtypes('category').columns) + list(X.select_dtypes('object').columns)

In [10]:
train_pool = Pool(X_train, label=y_train, cat_features=cat_features)

val_pool = Pool(X_val, label=y_val, cat_features=cat_features)

In [13]:
if get_gpu_device_count() > 0:
    task_type = 'GPU'
else:
    task_type = 'CPU'

params = {
    'iterations': 5000,
    'depth': 7, 
    'learning_rate': 0.05, 
    'random_state': 1,
    'eval_metric': 'AUC',
    'loss_function': 'Logloss',
    'auto_class_weights': 'Balanced',
    'task_type': task_type,
    'cat_features': cat_features
}

model = CatBoostClassifier(**params)

In [14]:
model.fit(
    train_pool, 
    eval_set=val_pool,
    use_best_model=True,
    verbose=10,
    early_stopping_rounds=50,
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9851441	best: 0.9851441 (0)	total: 55ms	remaining: 4m 34s
10:	test: 0.9878449	best: 0.9878449 (10)	total: 565ms	remaining: 4m 16s
20:	test: 0.9882740	best: 0.9882740 (20)	total: 1.07s	remaining: 4m 12s
30:	test: 0.9886535	best: 0.9886535 (30)	total: 1.57s	remaining: 4m 11s
40:	test: 0.9890065	best: 0.9890065 (40)	total: 2.07s	remaining: 4m 10s
50:	test: 0.9893953	best: 0.9893953 (50)	total: 2.56s	remaining: 4m 8s
60:	test: 0.9896947	best: 0.9896947 (60)	total: 3.08s	remaining: 4m 9s
70:	test: 0.9899992	best: 0.9899992 (70)	total: 3.64s	remaining: 4m 12s
80:	test: 0.9902392	best: 0.9902392 (80)	total: 4.14s	remaining: 4m 11s
90:	test: 0.9905048	best: 0.9905048 (90)	total: 4.63s	remaining: 4m 9s
100:	test: 0.9906707	best: 0.9906707 (100)	total: 5.11s	remaining: 4m 7s
110:	test: 0.9908095	best: 0.9908095 (110)	total: 5.58s	remaining: 4m 5s
120:	test: 0.9909689	best: 0.9909689 (120)	total: 6.05s	remaining: 4m 3s
130:	test: 0.9910853	best: 0.9910853 (130)	total: 6.54s	remaining: 

In [18]:
test_data = dataformer.construct_dataset(DatasetType.TEST)

In [19]:
X_test = test_data.drop('user_id').to_pandas()

In [20]:
predict = model.predict(X_test, prediction_type='RawFormulaVal')

In [23]:
from scipy.special import expit

In [25]:
probs = expit(predict)

In [31]:
import pandas as pd
import polars as pl

In [34]:
test_data.with_columns(pl.Series(probs).alias('predict')).select(('user_id', 'predict')).write_csv('overfitted_ahh.csv')